In [25]:
import os
from extractDataSrc.Lasco import Lasco
from extractDataSrc.InSitu import InSitu
from extractDataSrc.Eit195 import Eit195

In [2]:
destination_folders = ["data_processed/eit", "data_processed/mdi/mag", "data_processed/mdi/con" ,"data_processed/lasco/c2", "data_processed/lasco/c3", "data_processed/in_situ"]

for folder in destination_folders:
    os.makedirs(folder, exist_ok=True)

In [3]:
helper_folders = ["data_csv", "data_fits/eit"]

for folder in destination_folders:
    os.makedirs(folder, exist_ok=True)

In [ ]:
import os, shutil
folder = './data_processed/lasco/c2/'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))


start_datetime = '2003-10-19 00:00:00'
stop_datetime  =  '2003-11-09 23:59:00'
lasco = Lasco(start_datetime, stop_datetime)
lasco.extract_data("c2")

# Aktualny stav a dalsie kroky 30.7.2025

Co uz mam:
- Zadefinovane parametre definujuce _idealne_ CME
    - Izolovane, neiteragujuce, rychle, dobre definovana struktura (magneticky oblak (juzne/severne Bz)), silne Bz, Halo CME, mimo oblasti koronalnych dier
- Sposob ako detekovat ci je CME dobre definovany magneticky oblak
- Pomocou obrazkov z SOHO detekcia Halo CMEs

Pokracovanie:
- Ako filtrovat **interagujuce/neinteragujuce**, **ovlyvnene/neovylvnene CME**
    - CME-CME
    - CME-CH
        - CME-CIR, CME-HSS
- Ako filtrovat CME podla **CH aktivity**
    - Treba ich filtrovat na zaklade CH aktivity ?
    - Adam, **SCSS-Net**, **CH segmentation**
- Vytvorit procesy presnej identifikacie kazdeho parametra CME
- Vytvorit kompaktnu **vizualizaciu** vyobrazujucu vsetky potrebne parametre na identifikaciu **idealneho CME**
- Filtracia idealnych CME
    - Vytvorenie datasetu idealnych CME
- Predikcia idealneho CME
    - **DTW** (Dynamic Time Warping (Adam))
    - Reprodukcia Adamovho **GRU modelu**
    - **DTW** ako **loss function**
- Vysledky a dalsie upravy/vylepsienia modelu
    - CNN-GRU, CNN-LSTM, ...
- Dalsie kategorizacie CME, stromova struktura komplexnejsich CME...

# Ideal event definition

Ideal event should be an event that obeys by these characteristics:
- singular/isolated
    - No other CMEs in close time period
- non-interacting
    - Does **not** intercat strongly with CIRs, other CMEs, HSSs...
    - **TODO**: How to detect if CME is interacting with something/ its being altered
- fast
    - supersonic speeds (1000km/s)
- well defined magnetic cloud
    - CMEs have very smooth magnetic field, in helical shape
    - this can be observed in $B_z$ data as smooth transitions between polarities, also $|B|$ should be strong in the center of the cloud
- strong Bz and min{0, Bz}
- Halo CME
    - Coronograph shows halo around the sun, indicating CME coming towards earth
- sunspot region
    - Could be done by adams algorithm for extracting coronal holes from LASCO images
    - Regions of sun with large number or large coronal holes are not good for CME in terms of idealizations. CH produce HSS, fast solar wind, which impacts the CMEs, they are dragged, deflected, or slowed down. Something which would change the internal structure and is hard to predict. We will filter out CMEs which erupted in the region of CHs. 

## Filtration process

1. First we can decide if CME is well defined MC. We can do this by looking at $\mathbf{B}_z$, $T$ and $\beta$.
    - This will also tell us if southward $\mathbf{B}_z$ is strong and prolonged.
2. SOHO images for current event will tell us if there is HALO and if the CME is coming straight our way
3. $|\mathbf{V}|$ will tell us if the CME is fast enough ~1000km/s
4. With EIT images we will discard CMEs created in active CH region, this way we will eliminate any unpredictable interactions of CME.
5. Plotting DST will show us if there was any other CME or event occuring in the near past which could have interacted with our CME of interest


### We can filtrate with plots
One window of plots with basic time series of data. One window with SOHO and EIT images. 

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

%matplotlib qt

In [6]:
df = pd.read_csv('./data/aligned_event_9.csv')
df.index = pd.to_datetime(df['Timestamp'])

In [7]:
%matplotlib qt
df[['Dst', 'B_z']].plot()

<Axes: xlabel='Timestamp'>

## Adam paper Halo CME event dates

In [8]:
dates = [
    ['2000-07-05 00:00:00', '2000-07-25 23:59:00'],
    ['2001-10-26 00:00:00', '2001-11-13 23:59:00'],
    ['2001-11-14 00:00:00', '2001-12-04 23:59:00'],
    ['2002-05-12 00:00:00', '2002-05-31 23:59:00'],
    ['2003-10-19 00:00:00', '2003-11-09 23:59:00'],
    ['2003-11-10 00:00:00', '2003-11-30 23:59:00'],
    ['2005-09-01 00:00:00', '2005-09-21 23:59:00'],
    ['2006-12-04 00:00:00', '2006-12-24 23:59:00'],
    ['2011-08-02 00:00:00', '2011-08-15 23:59:00'],
    ['2012-02-28 00:00:00', '2012-03-19 23:59:00']
]
len(dates)

10

## Prepare code for full automatic visualisation needed for filtration process
- plots with all the needed parameters
- creating images of coronogrpah around the time CME hits earth

## Well defined Magnetic Cloud detection 

We can detect if CME is well defined magnetic cloud if the magnetic field $\mathbf{B}$ is behaving well while spacecraft is traveling through the CME. Magnetic cloud CME is defined by its flux rope. Flux rope could be tought of as an axis going through the whole CME where $\mathbf{B}$ is going along the axis as a coil, in a helical shape. 

Parameters that are to be satisfied for CME to resemble a Magnetic Cloud (MC) structure are to be found in the paper **Properties and geoeffectiveness of magnetic clouds in the rising, maximum and early declining phases of solar cycle 23** by K. E. J. Huttunen, R. Schwenn, V. Bothmer, and H. E. J. Koskinen. Here they are defiend as:
- $\mathbf{B}$ strength is higher than the average
- $\mathbf{B}$ direction rotates smoothly through a large angle
    - Depends on the orientation. Unipolar MCs have flux rope B oriented like ESW, WSE, WNE, ENW. This means that $\mathbf{B}_z$ southaward component is on the axis in the two of the cases of unipolar MCs. If $-\mathbf{B}_z$ is on the axis, it will have greater and prolonged impact, so greater CMEs.
- the proton temperature is low
- Because of the high magnetic field strength and low proton temperatures MCs have values of plasma beta significantly lower than 1
- average solar wind speed 420km/s


In [59]:
start = '2024-09-25 00:00:00'
end = '2024-10-15 23:59:00'

# omni = pd.read_csv("./data/omni_full_1964-2022.csv")
# omni.index = pd.to_datetime(omni['time1'])

In [4]:
omni[['BX_GSE', 'BY_GSE', 'BZ_GSE', 'DST', 'ABS_B', 'Beta']].loc['2000-07-10 00:00:00':'2000-07-20 00:00:00'].plot()

<Axes: xlabel='time1'>

In [60]:
insitu = InSitu(start, end)
insitu.extract_data("B", verbose=True)

transforming data...
saving data...
Done


In [61]:
B = pd.read_csv('./data_processed/in_situ/B_data.csv')
B.index = pd.to_datetime(B['Time'])
B.drop('Time', axis=1, inplace=True)
B['B'] = np.sqrt(B['B_x']**2 + B['B_y']**2 + B['B_z']**2)

In [62]:
B = B.resample('3min').mean()

### Get SYM_H data from HAPI

In [188]:
from extractDataSrc.Hapi import get_hapi_data
from hapiclient import hapi

server = 'https://cdaweb.gsfc.nasa.gov/hapi'
dataset = 'OMNI_HRO_1MIN'
parameter = 'SYM_H'

hapi_data = get_hapi_data(server, dataset, parameter, start, end)

In [64]:
SYM_data = []
for record in hapi_data:
    row = []
    row.append(record[0])
    row.append(round(record[1], 3))
    SYM_data.append(row)

SYM_H = pd.DataFrame(data=SYM_data, columns=["Time", 'SYM_H'])
SYM_H['Time'] = SYM_H['Time'].str.decode('utf-8')
SYM_H.index = pd.to_datetime(SYM_H['Time'], format="%Y-%m-%dT%H:%M:%S.000Z")
SYM_H.drop('Time', axis=1, inplace=True)
SYM_H = SYM_H.resample('3min').mean()

In [65]:
ts = SYM_H.idxmin().iloc[0]
after = ts + timedelta(hours=3)
before = ts - timedelta(hours=12)
after, before

(Timestamp('2024-10-11 02:12:00'), Timestamp('2024-10-10 11:12:00'))

In [66]:
plt.plot(SYM_H)
plt.plot(B['B_z'])

#### Get plasma beta and plasma temparature data

In [176]:
hapiT = get_hapi_data(server, 'OMNI_HRO2_1MIN', 'T,Beta', start, end)


In [183]:
T = []
for record in hapiT:
    row = []
    row.append(record[0])
    row.append(round(record[1], 3))
    row.append(round(record[2], 3))
    T.append(row)

T = pd.DataFrame(data=T, columns=["Time", 'T', 'Beta'])
T['Time'] = T['Time'].str.decode('utf-8')
T.index = pd.to_datetime(T['Time'], format="%Y-%m-%dT%H:%M:%S.000Z")
T.drop('Time', axis=1, inplace=True)
T['T'] = T['T'].replace([999.9, 9999.9, 9999999.0], np.nan)
T.loc[np.isclose(T["Beta"], 999.99), "Beta"] = np.nan
T = T.resample('3min').mean()

In [186]:
T.plot()


<Axes: xlabel='Time'>

In [155]:
info = hapi(server, "OMNI_HRO2_1MIN")
print(info)

{'HAPI': '2.0', 'resourceURL': 'https://cdaweb.gsfc.nasa.gov/misc/NotesO.html#OMNI_HRO2_1MIN', 'contact': 'J.H. King, N. Papatashvilli @ AdnetSystems, NASA GSFC', 'parameters': [{'name': 'Time', 'length': 24, 'units': 'UTC', 'type': 'isotime', 'fill': None}, {'name': 'IMF', 'description': 'OMNI ID code for the source spacecraft for time-shifted IMF values (see OMNI documentation link for codes)', 'units': None, 'type': 'integer', 'fill': '99'}, {'name': 'PLS', 'description': 'OMNI ID code for the source spacecraft  for time-shifted IP plasma values (see OMNI documentation link for codes)', 'units': None, 'type': 'integer', 'fill': '99'}, {'name': 'IMF_PTS', 'description': 'Number of fine time scale points in IMF averages', 'units': None, 'type': 'integer', 'fill': '999'}, {'name': 'PLS_PTS', 'description': 'Number of fine time scale points in plasma averages', 'units': None, 'type': 'integer', 'fill': '999'}, {'name': 'percent_interp', 'description': 'Percent interpolated', 'units': No

## Coronal holes region filtration

Coronal holes on sun are a birthplace of HSS (high speed streams). HSSs interact with slower solar wind, which results is SIRs. These can further interact with CMEs, or the HSSs can interact with CMEs themselves. Interactions vary, CMEs can be accelerated, slowed, deflected or deformed. These interactions impede the clean topologies of MCs, thus these ICMEs become more unpredictable. leading to undpredictable magnitudes of DST decreases, multiple decreases in short time and inSitu data which would not resemble any known pattern. That's why we will avoid CMEs erected in the regions of sun with high number or large % coverage of CHs. With this filtration we can say that:
- The ICMEs will not probably come deformed or adjusted, so we can expect nice MC.
- There wont be any CME-CME, CME-SIR, CME-HSS interactions.

In [2]:
import sunpy
from sunpy.net import attrs as a
from astropy.time import Time
import astropy.units as u
import sunpy.map
import scipy.ndimage as ndi
import albumentations
import cv2

C:\Users\Lukas\miniconda3\envs\space_lab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Download data

In [67]:
start_eit_time = Time('2000-07-14 08:00:00', scale='utc')
end_eit_time = Time('2000-07-14 10:00:00', scale='utc')

time_range = a.Time("2023-02-01 00:00", "2023-02-01 00:10")
result = sunpy.net.Fido.search(
    time_range,  # start and end time
    a.Instrument.aia,  # SDO/AIA instrument
    a.Wavelength(193*u.angstrom),  # EUV 193 Å channel
    a.Sample(5*u.minute) 
)

# Download the files
downloaded_files = sunpy.net.Fido.fetch(result)

print(downloaded_files)


Files Downloaded: 100%|████████████████████████████████████████████████████████████████| 2/2 [01:07<00:00, 33.97s/file]

['C:\\Users\\Lukas\\sunpy\\data\\aia.lev1.193A_2023_02_01T00_00_04.84Z.image_lev1.fits', 'C:\\Users\\Lukas\\sunpy\\data\\aia.lev1.193A_2023_02_01T00_05_04.84Z.image_lev1.fits']


### Plot multiple images

In [ ]:
eit_maps = []
for i in range(len(files)):
    if i % 2 != 0:
        eit_maps.append(sunpy.map.Map(files[i]))

# Determine the grid size for the plot
num_images = len(eit_maps)
nrows = int(num_images**0.5)
ncols = (num_images + nrows - 1) // nrows

# Create the subplot grid
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 12),
                         subplot_kw={'projection': eit_maps[0].wcs})

# Flatten the axes array for easier looping
axes = axes.flatten()

# Loop through the maps and plot them on their respective axes
for i, eit_map in enumerate(eit_maps):
    ax = axes[i]
    eit_map.plot(axes=ax, title=f"EIT 195 Å\n{eit_map.date.value}")
    
    # Optional: draw solar limb, grid, etc.
    eit_map.draw_limb(axes=ax, color='white', linestyle='--')
    ax.grid(False)

# Hide any unused subplots
for i in range(num_images, len(axes)):
    axes[i].axis('off')

plt.tight_layout()

### Predict mask for Coronal Holes on one image

#### Image augmentation

In [21]:
SDO_map = sunpy.map.Map(downloaded_files[1])

SDO_img = SDO_map.data

# resized_eit = ndi.zoom(eit_data, IMG_SIZE / max(eit_data.shape), order=1)
# normalized_eit = resized_eit / np.max(resized_eit)
# eit_map_array_image = np.array(normalized_eit)


'''
Normalize brigthness
'''
vmin = 0
vmax = 300
bright_SDO = np.clip(SDO_img, vmin, vmax)  # keep values within [vmin, vmax]
bright_SDO = ((bright_SDO - vmin) / (vmax - vmin) * 255).astype(np.uint8)

less_bright_SDO = np.clip(SDO_img, vmin, vmax + 200)
less_bright_SDO = ((less_bright_SDO - vmin) / ((vmax + 200) - vmin) * 255).astype(np.uint8)

'''
Crop image
'''

h, w = bright_SDO.shape
cx, cy = w // 2, h // 2   # center (SOHO images are centered already)
radius = min(cx, cy) - 400   # approximate disk radius

# make a circular mask
Y, X = np.ogrid[:h, :w]
dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
mask = dist <= radius

# apply mask
cropped_bright_SDO = np.full_like(bright_SDO, 255)  # fill with white background
cropped_bright_SDO[mask] = bright_SDO[mask]



'''
Resize image

CRB - cropped, resized, bright
'''
CRB_SDO = cv2.resize(cropped_bright_SDO, (256, 256), interpolation=cv2.INTER_AREA)


# normalised, cropped, resized, brighthened SDO, NCRB - SDO
pmin, pmax = np.percentile(CRB_SDO, (1, 99))
NCRB_SDO = np.clip((CRB_SDO - pmin) / (pmax - pmin), 0, 1)

NCRB_SDO = NCRB_SDO.reshape((1, 256, 256, 1))

# plt.imshow(NCRB_SDO[0], cmap='grey')
plt.imshow(less_bright_SDO)

#### Crop sun disk and make the back ground white

#### Model setup and CH prediction

In [11]:
from model_scss_net import scss_net
from metrics import dice_np, iou_np, dice, iou
from utils import plot_imgs, plot_metrics

model = scss_net( 
    (256, 256, 1),
    filters=32,       
    layers=4,
    batch_norm=True,
    drop_prob=0.5)

# Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",  
    metrics=[iou, dice])

In [12]:
model.load_weights("./model_ch_region.h5")

In [13]:
 y_pred = model.predict(NCRB_SDO)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [112]:
plot_imgs(imgs=NCRB_SDO, masks=y_pred, predictions=y_pred, n_imgs=1).show()

In [102]:
plt.imshow(augmented_NCRB_SDO[0], cmap='grey')
plt.imshow(y_pred[0], alpha=0.5)

In [22]:
pred_mask_resized = cv2.resize(y_pred[0], (4096, 4096), interpolation=cv2.INTER_NEAREST)

plt.imshow(less_bright_SDO, cmap='grey')
plt.imshow(pred_mask_resized, alpha=0.5)

# Fetch all needed omni data from sunpy

In [120]:
start_time = '2020-08-01'
end_time = '2020-08-30'

# Search OMNI 1-minute data (hro = high resolution)
result = sunpy.net.Fido.search(
    a.Time(start_time, end_time),
    a.Instrument('omni_hro_1min'),  # exact OMNI high-res instrument name
)

print(result)  # Check what files are available

# Download the files
files = sunpy.net.Fido.fetch(result)

if len(files) == 0:
    print("No files found for this query.")
else:
    # Load files into TimeSeries objects
    omni_ts_list = [ts.TimeSeries(f) for f in files]

    # If there is at least one TimeSeries, inspect columns
    print(omni_ts_list[0].columns)
    omni_ts_list[0].plot()

Results from 0 Providers:




Files Downloaded: 0file [00:00, ?file/s]

No files found for this query.
